In [1]:
import baskervillehall
import os, logging
import numpy as np
from kafka import KafkaConsumer, TopicPartition
import json
import time

from datetime import datetime, timedelta
from baskervillehall.model_io import ModelIO
from baskervillehall.baskervillehall_isolation_forest import BaskervillehallIsolationForest

In [2]:
kafka_url = ['kafka9-0.kafka9-headless.default.svc.cluster.local:9093','kafka9-1.kafka9-headless.default.svc.cluster.local:9093','kafka9-2.kafka9-headless.default.svc.cluster.local:9093']
topic = 'BASKERVILLEHALL_3'

os.environ['S3_ACCESS'] = 'c490f17bdb784fa08f4d11836ee18e48'
os.environ['S3_SECRET'] = 'e4b65e27b3734d0d96ce6038586ef43f'
os.environ['S3_ENDPOINT'] = 's3.gra.cloud.ovh.net'
os.environ['S3_REGION'] = 'GRA'
datetime_format = '%Y-%m-%d %H:%M:%S'

In [3]:
host = 'verafiles.org'

In [16]:
partitions = {
    'zhitomir.info': 1,
    'urban-pushkino.ru': 0,
    'dev.emawpb.org': 0,
    'palestinechronicle.com': 1,
    'equalit.ie': 0,
    'lexota.org': 0,
    'kavkaz-uzel.eu': 0,
    'amp.kavkaz-uzel.eu': 2,
    'indymedia.nl': 0,
    'btselem.org': 0,
    'verafiles.org': 1,
}

In [18]:
logger = logging.getLogger('pca')
logger.addHandler(logging.StreamHandler())
logger.setLevel('DEBUG')
s3_connection = {
    's3_access':os.environ['S3_ACCESS'],
    's3_secret':os.environ['S3_SECRET'],
    's3_endpoint': os.environ['S3_ENDPOINT'],
    's3_region':os.environ['S3_REGION']
}
model_io = ModelIO(**s3_connection, logger=logger)


In [7]:
datetime_format = '%Y-%m-%d %H:%M:%S'

In [9]:
model = model_io.load('s3://anton/baslervillehall/models24', host)
# model = model_io.load_exact_path(f's3://anton/baslervillehall/models18/{host}/2023_11_28___14_33.pkl')
# model = model_io.load_exact_path(f's3://anton/baslervillehall/models18/{host}/2023_11_29___00_38.pkl')

In [10]:
model

In [54]:
def read_sessions(host, start, stop, topic, kafka_url):
    partition = partitions[host]
    consumer = KafkaConsumer(
        bootstrap_servers=kafka_url,
        group_id='antonx_11'
    )
    ids = []
    num = 0
    
    print(f'Reading from kafka. Host = {host} ... from {start} to {stop} partition = {partition}')
    time_now = int(time.time())
    sessions = []

    consumer.assign([TopicPartition(topic, partition)])
    
    consumer.seek_to_beginning()
    # consumer.seek(TopicPartition(topic, partition), 96929044)
    
    complete = False
    while not complete:
        raw_messages = consumer.poll(timeout_ms=1000, max_records=5000)

        for topic_partition, messages in raw_messages.items():
            for message in messages:
                # prevent from getting messages too close to the current time
                # time_diff_in_minutes = (time_now - message.timestamp / 1000) / 60
                # if time_diff_in_minutes < 2:
                #     print(f'{time_diff_in_minutes} minutes. Topic offset is too close to the current times...')
                #     complete = True
                #     break
                
                if message.value is None :
                    continue
                if message.key is None:
                    continue
                if message.key.decode("utf-8") != host:
                    continue     

                value = json.loads(message.value.decode("utf-8"))
                # if value['ip'] == '35.204.46.217' and value['session_id'] == 'MDPfePcWYQkAAAAAZWWCsA':
                #     import pdb
                #     pdb.set_trace()
                session_stop = datetime.strptime(value['end'], datetime_format)
                if session_stop < start:
                    continue
                if session_stop > stop:
                    complete = True
                    print(f'{session_stop} > {stop}')
                    break
                
                # print(message)
                # import pdb
                # pdb.set_trace()
                
                sessions.append(value)

                num += 1
                if num % 100 == 0:
                    print(f'{num} sessions read', value['end'], message.timestamp)
    print(f'{len(sessions)} sessions read.')        
    return sessions


In [65]:
start = datetime(2023, 1, 4, 9, 50, 0, 0)
stop = datetime(2024, 1, 4, 9, 54, 30, 0)
sessions = read_sessions(host, start, stop, 'anton_sessions', 'kafkab-0.kafkab-headless.default.svc.cluster.local:9093,kafkab-2.kafkab-headless.default.svc.cluster.local:9093,kafkab-2.kafkab-headless.default.svc.cluster.local:9093')

Reading from kafka. Host = verafiles.org ... from 2023-01-04 09:50:00 to 2024-01-04 09:54:30 partition = 1
100 sessions read 2024-01-04 09:52:47 1704372346876
200 sessions read 2024-01-04 09:52:47 1704372348713
300 sessions read 2024-01-04 09:52:49 1704372350969
400 sessions read 2024-01-04 09:52:25 1704372353733
500 sessions read 2024-01-04 09:52:54 1704372357158
600 sessions read 2024-01-04 09:52:53 1704372359902
700 sessions read 2024-01-04 09:52:55 1704372362706
800 sessions read 2024-01-04 09:52:54 1704372366642
900 sessions read 2024-01-04 09:52:57 1704372370974
1000 sessions read 2024-01-04 09:53:01 1704372377355
1100 sessions read 2024-01-04 09:53:04 1704372382788
1200 sessions read 2024-01-04 09:53:07 1704372385845
1300 sessions read 2024-01-04 09:53:11 1704372391470
1400 sessions read 2024-01-04 09:53:19 1704372402502
1500 sessions read 2024-01-04 09:53:51 1704372405548
1600 sessions read 2024-01-04 09:53:56 1704372409376
1700 sessions read 2024-01-04 09:54:06 1704372413203
1

In [36]:
# start = datetime(2024, 1, 4, 9, 50, 0, 0)
# stop = datetime(2024, 1, 4, 9, 55, 0, 0)
# sessions = read_sessions(host, start, stop, topic, kafka_url)

Reading from kafka. Host = verafiles.org ... from 2024-01-04 09:50:00 to 2024-01-04 09:55:00 partition = 1
100 sessions read 2024-01-04 09:52:18 1704361965321
200 sessions read 2024-01-04 09:52:26 1704361972197
300 sessions read 2024-01-04 09:52:29 1704361980994
400 sessions read 2024-01-04 09:52:40 1704361988854
500 sessions read 2024-01-04 09:52:43 1704361998608
600 sessions read 2024-01-04 09:52:52 1704362013561
700 sessions read 2024-01-04 09:52:56 1704362027227
800 sessions read 2024-01-04 09:53:09 1704362051739
900 sessions read 2024-01-04 09:53:15 1704362062035
1000 sessions read 2024-01-04 09:53:16 1704362076096
1100 sessions read 2024-01-04 09:53:39 1704362084646
1200 sessions read 2024-01-04 09:53:54 1704362091658
1300 sessions read 2024-01-04 09:54:04 1704362100329
1400 sessions read 2024-01-04 09:54:19 1704362115400
2024-01-04 09:55:13 > 2024-01-04 09:55:00
1432 sessions read.


In [62]:
ips = set([(s['ip'], s['duration'], len(s['requests'])) for s in sessions])
print(f'Unique ips = {len(ips)}')

Unique ips = 1304


In [66]:
for s in sessions:
    if s['ip'] == '190.2.210.116':
        print(s)

{'host': 'verafiles.org', 'ua': 'SonyEricssonS500i/R6BC Browser/NetFront/3.3 Profile/MIDP-2.0 Configuration/CLDC-1.1', 'country': 'CO', 'session_id': '', 'ip': '190.2.210.116', 'start': '2024-01-04 09:52:53', 'end': '2024-01-04 09:52:53', 'duration': 0, 'fresh_sessions': False, 'requests': [{'ts': '2024-01-04 09:52:53', 'url': '/', 'ua': 'SonyEricssonS500i/R6BC Browser/NetFront/3.3 Profile/MIDP-2.0 Configuration/CLDC-1.1', 'query': 'end_date=1989-06-04&wvNwOfGTdu&guyStYWo8W', 'code': 405, 'type': 'text/html', 'payload': 0}]}
{'host': 'verafiles.org', 'ua': 'SonyEricssonK610i/R1CB Browser/NetFront/3.3 Profile/MIDP-2.0 Configuration/CLDC-1.1', 'country': 'CO', 'session_id': '-', 'ip': '190.2.210.116', 'start': '2024-01-04 09:52:53', 'end': '2024-01-04 09:52:53', 'duration': 0, 'fresh_sessions': False, 'requests': [{'ts': '2024-01-04 09:52:53', 'url': '/', 'ua': 'SonyEricssonK610i/R1CB Browser/NetFront/3.3 Profile/MIDP-2.0 Configuration/CLDC-1.1', 'query': 'end_date=1989-06-04&wvNwOfGTdu&

In [47]:
print(set([(s['session_id']) for s in sessions]))

{'', '-'}


In [25]:
model.datetime_format = datetime_format

In [26]:
scores = model.score_sessions(sessions)

In [27]:
scores[100:120]

array([-0.34804349, -0.32253933, -0.34630706, -0.34734845, -0.34155206,
       -0.33591821, -0.33026578, -0.32978875, -0.33590635, -0.33771857,
       -0.33026142, -0.34927146, -0.35780852, -0.21471447, -0.3375472 ,
       -0.32908843, -0.23354425, -0.29862243, -0.31931578, -0.35780852])

In [28]:
len(scores), len(scores[scores<0])

(2439, 2431)

In [29]:
print(f'Challenge rate = {len(scores[scores<0])*1.0/len(scores)}')

Challenge rate = 0.996719967199672


In [32]:
model_io._save_object(json.dumps(sessions, indent=2), 'anton/dataset/attacks/verafiles_2024_01_04.json')

In [13]:
duration = 60
hit_rate = 2
country = 'GB'
url = '/'
session = {'duration': duration, 'country': country, 'fresh_sessions':False, 'session_id':'-'}
requests = []
num_hits = int(duration * hit_rate / 60)
ts = datetime.now()
time_increment = 60.0 / hit_rate
countries = []

for i in range(num_hits):
    requests.append({'ts': ts, 
                     'url': f'{i}', #url, 
                     # 'url': url, 
                     'query': '', 'code': 200, 'type': 'text/html','payload': 1000})
    ts += timedelta(seconds=time_increment)
    countries.append(country)
session['requests'] = requests

ts = datetime.now()
score = model.score_sessions([session])
print(f'{(datetime.now() - ts).total_seconds()} seconds.')
print(score)

0.159829 seconds.
[-0.16685846]


In [9]:
num_hits

29

In [30]:
session_features = model.calculate_features_dict(session)

In [ ]:
session_features

In [17]:
model.feature_names


['request_rate',
 'response4xx_to_request_ratio',
 'top_page_to_request_ratio',
 'unique_path_to_request_ratio',
 ' path_depth_average',
 'fresh_session',
 'entropy']